In [19]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np


In [20]:
df=pd.read_csv("collegePlace_50000.csv")

df.head()

,Age,Gender,Stream,Internships,CGPA,Hostel,HistoryOfBacklogs,PlacedOrNot
0,22,Male,Information Technology,2,8,0,0,1
1,22,Male,Mechanical,2,7,1,0,0
2,21,Male,Computer Science,0,8,0,0,1
3,22,Male,Civil,0,7,1,0,0
4,21,Male,Computer Science,1,8,0,0,1


In [21]:
df.head()
df.columns.value_counts()

Age                  1
Gender               1
Stream               1
Internships          1
CGPA                 1
Hostel               1
HistoryOfBacklogs    1
PlacedOrNot          1
Name: count, dtype: int64

In [22]:
#bool_cols = df.select_dtypes(include='bool').columns

#df[bool_cols] = df[bool_cols].astype(int)
#from sklearn.preprocessing import LabelEncoder,StandardScaler

#le=LabelEncoder()
#df["Gender"]=le.fit_transform(df["Gender"])
#df["Stream"]=le.fit_transform(df["Stream"])
# One-hot encode categorical features
df = pd.get_dummies(df, columns=["Stream", "Gender"], drop_first=True)

# Convert the resulting boolean columns to integers (0 or 1)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [23]:
df.dtypes


Age                                     int64
Internships                             int64
CGPA                                    int64
Hostel                                  int64
HistoryOfBacklogs                       int64
PlacedOrNot                             int64
Stream_Computer Science                 int64
Stream_Electrical                       int64
Stream_Electronics And Communication    int64
Stream_Information Technology           int64
Stream_Mechanical                       int64
Gender_Male                             int64
dtype: object

In [24]:
df.head()
x=df.drop("PlacedOrNot",axis=1)
y=df["PlacedOrNot"]


In [25]:
#train test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [26]:
#scaling
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)


In [27]:
#tensors
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.long)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.long)

In [28]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)

In [29]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle="True")
test_loader = DataLoader(test_dataset,batch_size=32)

In [30]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
            nn.Linear(x_train_tensor.shape[1],64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,2)
            
        )
    def forward (self,x):
        return self.model(x)

In [31]:
model=ANN()
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.0005)

In [32]:
epochs=200
for epoch in range (epochs):
    model.train()
    running_loss=0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/200, loss = 0.38122079641819
epoch = 2/200, loss = 0.30786809841394425
epoch = 3/200, loss = 0.289548294454813
epoch = 4/200, loss = 0.27593161948025224
epoch = 5/200, loss = 0.2657145040988922
epoch = 6/200, loss = 0.25743420566022396
epoch = 7/200, loss = 0.25347254266142843
epoch = 8/200, loss = 0.2500419688522816
epoch = 9/200, loss = 0.24368317630887032
epoch = 10/200, loss = 0.23975063268244268
epoch = 11/200, loss = 0.23766510163843632
epoch = 12/200, loss = 0.23454572384655475
epoch = 13/200, loss = 0.23252272050082684
epoch = 14/200, loss = 0.22932602854073048
epoch = 15/200, loss = 0.22809616063833238
epoch = 16/200, loss = 0.2257570129364729
epoch = 17/200, loss = 0.22471439772844315
epoch = 18/200, loss = 0.2238082855582237
epoch = 19/200, loss = 0.22232289886176587
epoch = 20/200, loss = 0.22101179408580066
epoch = 21/200, loss = 0.21879893935024738
epoch = 22/200, loss = 0.21822362646460533
epoch = 23/200, loss = 0.21748638248741628
epoch = 24/200, loss = 0.2165

In [33]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(yb.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

print("Accuracy:", accuracy_score(y_true, y_pred) * 100)

print("\nConfusion Matrix")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report")
print(classification_report(y_true, y_pred))

Accuracy: 92.01

Confusion Matrix
[[4352  123]
 [ 676 4849]]

Classification Report
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      4475
           1       0.98      0.88      0.92      5525

    accuracy                           0.92     10000
   macro avg       0.92      0.93      0.92     10000
weighted avg       0.93      0.92      0.92     10000



In [34]:
model.eval()

test_loss = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        loss = criteria(outputs, yb)
        test_loss += loss.item()

print("Test Loss:", test_loss / len(test_loader))

Test Loss: 0.17348849968597913


In [35]:
import pickle

# 1. Save the PyTorch Model Weights
torch.save(model.state_dict(), "placement_model.pth")

# 2. Save the StandardScaler instance
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and Scaler saved successfully!")

Model and Scaler saved successfully!


In [36]:
df.columns

Index(['Age', 'Internships', 'CGPA', 'Hostel', 'HistoryOfBacklogs',
       'PlacedOrNot', 'Stream_Computer Science', 'Stream_Electrical',
       'Stream_Electronics And Communication', 'Stream_Information Technology',
       'Stream_Mechanical', 'Gender_Male'],
      dtype='str')